Imports

In [6]:
import pandas as pd

Read Raw Data

In [7]:
mbs_raw_data_df = pd.read_csv('../data/raw/mbs_dataset_raw_sample.csv', 
                              dtype={"tax_ref_no": str,"natlidno": str,"mobilephone": str, "pr_mbr_no": str}
                              )

Validation

In [8]:
# Pre-Validation

from validation import validate_all

validate_all(mbs_raw_data_df)

,pyrl_dt,product_type,case_mbr_key,case_key,cont_no,plan_nm,mbr_no,pr_mbr_no,natlidno,passport_no,...,savings_pot_bal,retirement_pot_bal,vested_pot_bal,tlaa_prov_bal,trading_fund,moderate,conservative,growth,acc_credit,opening_balance
0,2024/12/31,SACCAWU,1889056,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,B444365B,33392662,8503025747085,NaN,...,1535.52,2308.95,41495.28,69690.82,856.40,0.0,0.0,114174.17,115030.57,NaN
1,2024/12/31,SACCAWU,6938270,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,S415149A,NaN,9309230207082,NaN,...,1108.61,1295.74,4655.04,0.00,564.79,0.0,0.0,6494.60,7059.39,NaN
2,2024/12/31,SACCAWU,316119,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,A286555B,33370557,7905170471083,NaN,...,6436.62,2515.72,48520.88,136664.40,933.09,0.0,0.0,193204.53,194137.62,NaN
3,2024/12/31,SACCAWU,4751992,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,10100465,41043770,8212270620084,NaN,...,1013.05,1934.99,32091.79,16829.47,717.71,0.0,0.0,51151.59,51869.30,NaN
4,2024/12/31,SACCAWU,4634801,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,10009725,41033093,9609165669085,NaN,...,741.47,1319.76,21474.28,11302.06,527.85,0.0,0.0,34309.72,34837.57,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026/07/31,SACCAWU,4751992,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,10100465,41043770,8212270620084,NaN,...,276.63,553.35,0.00,0.00,829.98,NaN,NaN,NaN,NaN,NaN
96,2026/07/31,SACCAWU,4634801,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,10009725,41033093,9609165669085,NaN,...,203.08,406.22,0.00,0.00,609.30,NaN,NaN,NaN,NaN,NaN
97,2026/07/31,SACCAWU,1889056,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,B444365B,33392662,8503025747085,NaN,...,324.06,648.20,0.00,0.00,972.26,NaN,NaN,NaN,NaN,NaN
98,2026/07/31,SACCAWU,316119,2591,S001280D,PICK 'N PAY RETAILERS (PTY) LTD,A286555B,33370557,7905170471083,NaN,...,350.27,700.65,0.00,0.00,1050.92,NaN,NaN,NaN,NaN,NaN


Read Validated Data

In [9]:
mbs_data_df = pd.read_csv('../data/primary/mbs_dataset_validated.csv', 
                            dtype={"tax_ref_no": str, "natlidno": str, "mobilephone": str, "pr_mbr_no": str}
                            )

mbs_data_df = mbs_raw_data_df # Remove once validation code is complete

Generate Statement for each Member

In [10]:
from pathlib import Path

from PIL import Image
from statement_templates.saccawu import generate

old_mutual_logo = Image.open("../assets/old_mutual_header.png")
saccawu_logo = Image.open("../assets/saccawu_logo.png")

reporting_dt = mbs_data_df[mbs_data_df["acc_credit"].notna()]["pyrl_dt"].max()
reporting_dt = pd.to_datetime(reporting_dt)

start_dt = reporting_dt.replace(day=1) - pd.DateOffset(months=11)

# Loop through members to process each members statement
for case_mbr_key in mbs_data_df["case_mbr_key"].dropna().unique():
    
    # Filter mbs data between a period and current member
    mbs_data_df["pyrl_dt"] = pd.to_datetime(mbs_data_df["pyrl_dt"])
    mbs_data = mbs_data_df[mbs_data_df["pyrl_dt"].between(start_dt, reporting_dt) & (mbs_data_df["case_mbr_key"] == case_mbr_key)]
        
    output_path = Path(f"../statements/{case_mbr_key}_member_statement.pdf")

    generate.generate_statement(mbs_data, output_path, old_mutual_logo, saccawu_logo, reporting_dt, start_dt)